# Système RAG pour l'Exploitation de Textes Juridiques
## Code de la Route Marocain

**Université Abdelmalek Essaadi — Faculté des Sciences et Techniques de Tanger**  
**Master IASD 2026 | Devoir 2 — NLP**  
**Pr. I. BENABDELOUAHAB**

---

## Introduction : Qu'est-ce que le RAG ?

**Retrieval-Augmented Generation (RAG)** est une architecture hybride qui combine deux composants fondamentaux :

1. **Le Retriever (Système de Récupération)** : Indexe un corpus de documents et, face à une question, sélectionne les passages les plus pertinents par similarité sémantique ou lexicale.

2. **Le Generator (Générateur de Réponse)** : Prend les documents récupérés comme contexte et génère une réponse cohérente en langage naturel.

### Pourquoi utiliser RAG pour le droit ?

Les textes juridiques sont précis, structurés, et exigent une fidélité absolue aux sources. Le RAG garantit que les réponses sont **ancrées dans les documents officiels** et non inventées par le modèle (hallucination). C'est particulièrement crucial pour le Code de la Route marocain où une erreur peut avoir des conséquences légales.

### Architecture du Système

```
Question Utilisateur
        │
        ▼
┌─────────────────┐
│  Détection      │  ← Hors domaine ?
│  Hors Domaine   │
└────────┬────────┘
         │ Non
         ▼
┌─────────────────┐
│   RETRIEVER     │
│  TF-IDF + FAISS │  ← Recherche hybride
│   (Hybrid)      │
└────────┬────────┘
         │ Top-K documents
         ▼
┌─────────────────┐
│ Prompt Builder  │  ← Contexte + Question
└────────┬────────┘
         │
         ▼
┌─────────────────┐
│  LLM Simulé     │  ← Génère la réponse
└────────┬────────┘
         │
         ▼
   Réponse Finale
   + Références
```

---
## 📦 Installation des Dépendances

In [1]:
# Installation des bibliothèques nécessaires
!pip install sentence-transformers faiss-cpu scikit-learn pandas numpy rouge-score gradio -q

---
## 1. 📂 Préparation des Données

Cette section charge le fichier CSV contenant les articles juridiques du Code de la Route, effectue un nettoyage et une normalisation des textes arabes, puis découpe les données en unités exploitables (*chunks*).

In [2]:
import pandas as pd
import numpy as np
import re
import unicodedata
import warnings
warnings.filterwarnings('ignore')

# ─────────────────────────────────────────────
# 1.1 Chargement du fichier CSV
# ─────────────────────────────────────────────
CSV_PATH = "C:/Users/hp/Desktop/export_final.csv"  # Mettre à jour le chemin si nécessaire

df = pd.read_csv(CSV_PATH)
print(f"✅ Données chargées : {df.shape[0]} lignes, {df.shape[1]} colonnes")
print(f"\n📋 Colonnes : {df.columns.tolist()}")
df.head(3)

✅ Données chargées : 529 lignes, 10 colonnes

📋 Colonnes : ['article_id', 'infraction_desc', 'type_article', 'categorie_vehicule', 'amende_fixe', 'points_retrait', 'mots_cles', 'has_sanction', 'has_amende', 'has_points']


,article_id,infraction_desc,type_article,categorie_vehicule,amende_fixe,points_retrait,mots_cles,has_sanction,has_amende,has_points
0,1,المادة 1 ال يجوز ءلي شخص ءن يسوق مركبة ذات محر...,autre,non_precise,NaN,NaN,aucun,False,False,False
1,2,المادة 2 استثناا من ءحكام المادة اءلولى ءعاله ...,autre,non_precise,NaN,NaN,permis,False,False,False
2,3,المادة 3 يجب على الساءقين الحاصلين على رخصة سي...,obligation,non_precise,NaN,NaN,permis,False,False,False


In [3]:
# ─────────────────────────────────────────────
# 1.2 Exploration initiale des données
# ─────────────────────────────────────────────
print("=" * 60)
print("STATISTIQUES DESCRIPTIVES")
print("=" * 60)
print(f"\n📊 Types d'articles :")
print(df['type_article'].value_counts())

print(f"\n🚗 Catégories de véhicules :")
print(df['categorie_vehicule'].value_counts())

print(f"\n💰 Articles avec amende : {df['has_amende'].sum()}")
print(f"⚠️  Articles avec sanction : {df['has_sanction'].sum()}")
print(f"🔢 Articles avec retrait de points : {df['has_points'].sum()}")
print(f"\n❓ Valeurs manquantes :")
print(df.isnull().sum())

STATISTIQUES DESCRIPTIVES

📊 Types d'articles :
type_article
autre         291
sanction      113
obligation    108
definition     17
Name: count, dtype: int64

🚗 Catégories de véhicules :
categorie_vehicule
non_precise                            492
poids_lourd                             19
voiture_legere                           6
transport_commun                         3
transport_commun, poids_lourd            2
moto, voiture_legere, poids_lourd        2
moto, transport_commun, poids_lourd      1
moto                                     1
moto, poids_lourd                        1
moto, voiture_legere                     1
voiture_legere, poids_lourd              1
Name: count, dtype: int64

💰 Articles avec amende : 29
⚠️  Articles avec sanction : 113
🔢 Articles avec retrait de points : 77

❓ Valeurs manquantes :
article_id              0
infraction_desc         0
type_article            0
categorie_vehicule      0
amende_fixe           500
points_retrait        452
mots_cles    

In [4]:
# ─────────────────────────────────────────────
# 1.3 Nettoyage et normalisation du texte
# ─────────────────────────────────────────────

def normalize_arabic_text(text: str) -> str:
    """
    Nettoie et normalise un texte arabe :
    - Normalise les caractères Unicode
    - Supprime les diacritiques (tashkeel)
    - Normalise les alifs, waw, ya
    - Supprime les espaces multiples
    - Supprime les caractères non-arabes/non-pertinents
    """
    if not isinstance(text, str):
        return ""

    # 1. Normalisation Unicode NFC
    text = unicodedata.normalize('NFC', text)

    # 2. Suppression des diacritiques arabes (تشكيل)
    arabic_diacritics = re.compile(r'[\u064B-\u065F\u0670]')
    text = arabic_diacritics.sub('', text)

    # 3. Normalisation des variantes de Alif (ء ا أ إ آ → ا)
    text = re.sub(r'[أإآء]', 'ا', text)

    # 4. Normalisation du Ya (ى → ي)
    text = re.sub(r'ى', 'ي', text)

    # 5. Normalisation du Waw Hamza (ؤ → و)
    text = re.sub(r'ؤ', 'و', text)

    # 6. Suppression des caractères spéciaux sauf ponctuation de base
    text = re.sub(r'[^\u0600-\u06FF\u0750-\u077F\s0-9a-zA-Z.,;:()\-]', ' ', text)

    # 7. Suppression des espaces multiples
    text = re.sub(r'\s+', ' ', text).strip()

    return text


def clean_keyword(kw: str) -> str:
    """Normalise les mots-clés français."""
    if not isinstance(kw, str) or kw == 'aucun':
        return ''
    return kw.replace('"', '').strip()


# Application du nettoyage
print("🔄 Nettoyage des textes en cours...")

df_clean = df.copy()
df_clean['infraction_desc_clean'] = df_clean['infraction_desc'].apply(normalize_arabic_text)
df_clean['mots_cles_clean'] = df_clean['mots_cles'].apply(clean_keyword)

# Suppression des lignes avec texte vide après nettoyage
df_clean = df_clean[df_clean['infraction_desc_clean'].str.len() > 10].copy()
df_clean = df_clean.reset_index(drop=True)

print(f"✅ Nettoyage terminé : {len(df_clean)} articles valides")
print(f"\n🔍 Exemple avant : {df['infraction_desc'].iloc[0][:80]}...")
print(f"🔍 Exemple après : {df_clean['infraction_desc_clean'].iloc[0][:80]}...")

🔄 Nettoyage des textes en cours...
✅ Nettoyage terminé : 528 articles valides

🔍 Exemple avant : المادة 1 ال يجوز ءلي شخص ءن يسوق مركبة ذات محرك ءو مجموعة مركبات على الطريق العم...
🔍 Exemple après : المادة 1 ال يجوز الي شخص ان يسوق مركبة ذات محرك او مجموعة مركبات علي الطريق العم...


In [5]:
# ─────────────────────────────────────────────
# 1.4 Découpage en chunks (unités exploitables)
# ─────────────────────────────────────────────
# Stratégie : chaque article de loi = 1 chunk
# Pour les articles longs, on effectue un découpage supplémentaire

MAX_CHUNK_CHARS = 500  # Longueur maximale d'un chunk

def create_chunk(row: pd.Series) -> dict:
    """
    Crée un chunk structuré à partir d'une ligne du DataFrame.
    Chaque chunk contient :
    - Le texte de l'article
    - Les métadonnées (ID article, type, catégorie, amendes, points)
    """
    # Construire le texte enrichi du chunk
    metadata_text = f"[Article {row['article_id']}]"
    if row.get('mots_cles_clean', ''):
        metadata_text += f" [Mots-clés: {row['mots_cles_clean']}]"
    if pd.notna(row.get('amende_fixe')) and row.get('amende_fixe', 0) > 0:
        metadata_text += f" [Amende: {row['amende_fixe']} MAD]"
    if pd.notna(row.get('points_retrait')) and row.get('points_retrait', 0) > 0:
        metadata_text += f" [Points retirés: {int(row['points_retrait'])}]"

    full_text = f"{metadata_text} {row['infraction_desc_clean']}"

    return {
        'chunk_id': f"art_{row['article_id']}_{row.name}",
        'article_id': row['article_id'],
        'text': row['infraction_desc_clean'],
        'enriched_text': full_text,
        'type_article': row.get('type_article', 'autre'),
        'categorie_vehicule': row.get('categorie_vehicule', 'non_precise'),
        'amende_fixe': row.get('amende_fixe', None),
        'points_retrait': row.get('points_retrait', None),
        'mots_cles': row.get('mots_cles_clean', ''),
        'has_sanction': row.get('has_sanction', False),
        'has_amende': row.get('has_amende', False),
    }


def split_long_chunk(chunk: dict, max_chars: int = MAX_CHUNK_CHARS) -> list:
    """
    Découpe un chunk long en sous-chunks si le texte dépasse max_chars.
    Préserve les métadonnées dans chaque sous-chunk.
    """
    text = chunk['text']
    if len(text) <= max_chars:
        return [chunk]

    # Découper par phrase (séparateurs possibles en arabe et français)
    sentences = re.split(r'(?<=[.،؛])\s+', text)
    sub_chunks = []
    current = ""
    sub_idx = 0

    for sent in sentences:
        if len(current) + len(sent) > max_chars and current:
            sub_chunk = chunk.copy()
            sub_chunk['chunk_id'] = f"{chunk['chunk_id']}_p{sub_idx}"
            sub_chunk['text'] = current.strip()
            sub_chunk['enriched_text'] = f"[Article {chunk['article_id']}] {current.strip()}"
            sub_chunks.append(sub_chunk)
            current = sent
            sub_idx += 1
        else:
            current += " " + sent if current else sent

    if current:
        sub_chunk = chunk.copy()
        sub_chunk['chunk_id'] = f"{chunk['chunk_id']}_p{sub_idx}"
        sub_chunk['text'] = current.strip()
        sub_chunk['enriched_text'] = f"[Article {chunk['article_id']}] {current.strip()}"
        sub_chunks.append(sub_chunk)

    return sub_chunks if sub_chunks else [chunk]


# Construction de la base de chunks
all_chunks = []
for _, row in df_clean.iterrows():
    chunk = create_chunk(row)
    sub_chunks = split_long_chunk(chunk)
    all_chunks.extend(sub_chunks)

chunks_df = pd.DataFrame(all_chunks)

print(f"✅ {len(chunks_df)} chunks créés à partir de {len(df_clean)} articles")
print(f"📏 Longueur moyenne d'un chunk : {chunks_df['text'].str.len().mean():.0f} caractères")
print(f"📏 Longueur max : {chunks_df['text'].str.len().max()} | Min : {chunks_df['text'].str.len().min()}")
chunks_df.head(3)

✅ 528 chunks créés à partir de 528 articles
📏 Longueur moyenne d'un chunk : 138 caractères
📏 Longueur max : 269 | Min : 12


,chunk_id,article_id,text,enriched_text,type_article,categorie_vehicule,amende_fixe,points_retrait,mots_cles,has_sanction,has_amende
0,art_1_0,1,المادة 1 ال يجوز الي شخص ان يسوق مركبة ذات محر...,[Article 1] المادة 1 ال يجوز الي شخص ان يسوق م...,autre,non_precise,NaN,NaN,,False,False
1,art_2_1,2,المادة 2 استثناا من احكام المادة االولي اعاله ...,[Article 2] [Mots-clés: permis] المادة 2 استثن...,autre,non_precise,NaN,NaN,permis,False,False
2,art_3_2,3,المادة 3 يجب علي السااقين الحاصلين علي رخصة سي...,[Article 3] [Mots-clés: permis] المادة 3 يجب ع...,obligation,non_precise,NaN,NaN,permis,False,False


---
## 2. 🔍 Indexation et Recherche (Retriever)

Le retriever est le cœur du système RAG. Son rôle est de trouver les documents les plus pertinents pour répondre à une question.

Nous implémentons **deux méthodes complémentaires** :
- **TF-IDF** : Recherche lexicale rapide basée sur la fréquence des termes
- **SentenceTransformers + FAISS** : Recherche sémantique basée sur la signification

### Pourquoi la recherche hybride ?
La recherche lexicale (TF-IDF) est précise pour les termes exacts (numéros d'articles, termes juridiques spécifiques), mais rate les synonymes. La recherche sémantique comprend le sens, mais peut manquer de précision sur des termes techniques. La combinaison des deux offre le meilleur des deux mondes.

In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
import faiss
import pickle

# ─────────────────────────────────────────────
# 2.1 Retriever TF-IDF
# ─────────────────────────────────────────────

print("🔄 Construction de l'index TF-IDF...")

# Utilisation du texte enrichi pour la vectorisation TF-IDF
corpus = chunks_df['enriched_text'].tolist()

tfidf_vectorizer = TfidfVectorizer(
    analyzer='char_wb',       # Analyse au niveau des caractères (meilleur pour l'arabe)
    ngram_range=(2, 4),       # N-grammes de caractères
    max_features=20000,       # Vocabulaire limité
    sublinear_tf=True,        # Lissage logarithmique
    min_df=1,                 # Fréquence minimale
)

tfidf_matrix = tfidf_vectorizer.fit_transform(corpus)
print(f"✅ Index TF-IDF construit : matrice {tfidf_matrix.shape}")


def retrieve_tfidf(query: str, top_k: int = 5) -> list:
    """
    Retriever TF-IDF :
    1. Vectorise la question
    2. Calcule la similarité cosinus avec tous les chunks
    3. Retourne les top_k chunks les plus proches

    Args:
        query: Question de l'utilisateur
        top_k: Nombre de documents à retourner

    Returns:
        Liste de dicts contenant le chunk et son score
    """
    query_clean = normalize_arabic_text(query)
    query_vec = tfidf_vectorizer.transform([query_clean])
    scores = cosine_similarity(query_vec, tfidf_matrix).flatten()
    top_indices = np.argsort(scores)[::-1][:top_k]

    results = []
    for idx in top_indices:
        if scores[idx] > 0:
            result = chunks_df.iloc[idx].to_dict()
            result['tfidf_score'] = float(scores[idx])
            results.append(result)
    return results


# Test rapide
test_results = retrieve_tfidf("رخصة السياقة", top_k=3)
print(f"\n🧪 Test TF-IDF pour 'رخصة السياقة' :")
for r in test_results:
    print(f"  Article {r['article_id']} | Score: {r['tfidf_score']:.4f} | {r['text'][:60]}...")

🔄 Construction de l'index TF-IDF...
✅ Index TF-IDF construit : matrice (528, 11846)

🧪 Test TF-IDF pour 'رخصة السياقة' :
  Article 37 | Score: 0.3850 | المادة 37 يجب ان يتضمن الحالم المحررة فيه رخصة السياقة ، علي...
  Article 228 | Score: 0.3200 | المادة 228 اعاله، اذا كان حالم رخصة السياقة يمكن من تسجيل ال...
  Article 166 | Score: 0.3171 | المادة 166 1 اعاله، لتوقيف رخصة السياقة ملدة سنة الي سنتين. ...


In [8]:
# ─────────────────────────────────────────────
# 2.2 Retriever Sémantique (SentenceTransformers + FAISS)
# ─────────────────────────────────────────────
print("🔄 Chargement du modèle d'embeddings...")

# Modèle multilingue optimisé pour l'arabe et le français
# paraphrase-multilingual-MiniLM-L12-v2 : léger, multilingue, performant
EMBEDDING_MODEL = 'paraphrase-multilingual-MiniLM-L12-v2'
embedding_model = SentenceTransformer(EMBEDDING_MODEL)

print(f"✅ Modèle chargé : {EMBEDDING_MODEL}")
print(f"   Dimension des embeddings : {embedding_model.get_sentence_embedding_dimension()}")

# Génération des embeddings pour tous les chunks
print("\n🔄 Génération des embeddings (peut prendre quelques minutes)...")
embeddings = embedding_model.encode(
    corpus,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True  # Normalisation L2 pour la similarité cosinus
)

print(f"✅ Embeddings générés : shape {embeddings.shape}")

🔄 Chargement du modèle d'embeddings...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Modèle chargé : paraphrase-multilingual-MiniLM-L12-v2
   Dimension des embeddings : 384

🔄 Génération des embeddings (peut prendre quelques minutes)...


Batches:   0%|          | 0/9 [00:00<?, ?it/s]

✅ Embeddings générés : shape (528, 384)


In [9]:
# ─────────────────────────────────────────────
# 2.3 Construction de l'index FAISS
# ─────────────────────────────────────────────
print("🔄 Construction de l'index FAISS...")

embedding_dim = embeddings.shape[1]

# Index FAISS avec produit scalaire (≡ similarité cosinus si embeddings normalisés)
faiss_index = faiss.IndexFlatIP(embedding_dim)
faiss_index.add(embeddings.astype(np.float32))

print(f"✅ Index FAISS construit")
print(f"   Nombre de vecteurs indexés : {faiss_index.ntotal}")
print(f"   Dimension des vecteurs : {embedding_dim}")


def retrieve_semantic(query: str, top_k: int = 5) -> list:
    """
    Retriever Sémantique basé sur SentenceTransformers + FAISS :
    1. Encode la question en vecteur dense
    2. Recherche les k plus proches voisins dans FAISS
    3. Retourne les chunks correspondants avec leurs scores

    Args:
        query: Question de l'utilisateur
        top_k: Nombre de documents à retourner

    Returns:
        Liste de dicts contenant le chunk et son score de similarité
    """
    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True,
        convert_to_numpy=True
    ).astype(np.float32)

    # Recherche FAISS des top_k voisins
    scores, indices = faiss_index.search(query_embedding, top_k)

    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx >= 0 and score > 0:
            result = chunks_df.iloc[idx].to_dict()
            result['semantic_score'] = float(score)
            results.append(result)
    return results


# Test rapide
test_results_sem = retrieve_semantic("ما هي عقوبة السياقة بدون رخصة", top_k=3)
print(f"\n🧪 Test sémantique pour 'عقوبة السياقة بدون رخصة' :")
for r in test_results_sem:
    print(f"  Article {r['article_id']} | Score: {r['semantic_score']:.4f} | {r['text'][:60]}...")

🔄 Construction de l'index FAISS...
✅ Index FAISS construit
   Nombre de vecteurs indexés : 528
   Dimension des vecteurs : 384

🧪 Test sémantique pour 'عقوبة السياقة بدون رخصة' :
  Article 155 | Score: 0.8033 | المادة 155 يعاقب بغرامة من الفين 2.000 الي خمسة ااالف 5.000 ...
  Article 149 | Score: 0.7718 | المادة 149 بعده، يعاقب بغرامة من الفين 2.000 الي اربعة ااالف...
  Article 152 | Score: 0.7538 | المادة 152 1 يعاقب بغرامة من الف 1.000 درهم الي اربعة ااالف ...


In [10]:
# ─────────────────────────────────────────────
# 2.4 Retriever Hybride (TF-IDF + FAISS)
# ─────────────────────────────────────────────
# La recherche hybride combine les scores des deux méthodes
# via une moyenne pondérée (Reciprocal Rank Fusion)

def reciprocal_rank_fusion(results_list: list, k: int = 60) -> list:
    """
    Reciprocal Rank Fusion (RRF) : algorithme standard pour fusionner
    plusieurs listes de résultats de recherche.
    Score RRF d'un document à la position r : 1 / (k + r)

    Args:
        results_list: Liste de listes de résultats (chaque liste provient d'un retriever)
        k: Constante RRF (typiquement 60)

    Returns:
        Documents fusionnés et re-classés par score RRF
    """
    scores = {}
    doc_store = {}

    for results in results_list:
        for rank, doc in enumerate(results):
            doc_id = doc['chunk_id']
            scores[doc_id] = scores.get(doc_id, 0) + 1.0 / (k + rank + 1)
            doc_store[doc_id] = doc

    # Tri par score RRF décroissant
    sorted_ids = sorted(scores, key=scores.get, reverse=True)

    fused_results = []
    for doc_id in sorted_ids:
        doc = doc_store[doc_id].copy()
        doc['hybrid_score'] = scores[doc_id]
        fused_results.append(doc)

    return fused_results


def get_relevant_documents(question: str, top_k: int = 5) -> list:
    """
    🔑 FONCTION PRINCIPALE DU RETRIEVER HYBRIDE

    Combine TF-IDF et recherche sémantique via RRF pour retourner
    les top_k documents les plus pertinents pour une question donnée.

    Args:
        question: Question en langage naturel (arabe ou français)
        top_k: Nombre de documents à retourner

    Returns:
        Liste triée des documents les plus pertinents avec leurs scores
    """
    # Récupération via les deux retrievers
    tfidf_results = retrieve_tfidf(question, top_k=top_k * 2)
    semantic_results = retrieve_semantic(question, top_k=top_k * 2)

    # Fusion hybride via RRF
    hybrid_results = reciprocal_rank_fusion([tfidf_results, semantic_results])

    return hybrid_results[:top_k]


# ─────── Test comparatif ───────
test_query = "ما هي شروط الحصول على رخصة السياقة"
print(f"🧪 TEST COMPARATIF pour : '{test_query}'")
print("=" * 70)

tfidf_r = retrieve_tfidf(test_query, top_k=3)
sem_r = retrieve_semantic(test_query, top_k=3)
hybrid_r = get_relevant_documents(test_query, top_k=3)

print("\n📊 TF-IDF Top-3 :")
for r in tfidf_r:
    print(f"  [Art. {r['article_id']}] Score={r['tfidf_score']:.4f} | {r['text'][:50]}...")

print("\n🧠 Sémantique Top-3 :")
for r in sem_r:
    print(f"  [Art. {r['article_id']}] Score={r['semantic_score']:.4f} | {r['text'][:50]}...")

print("\n🔀 Hybride (RRF) Top-3 :")
for r in hybrid_r:
    print(f"  [Art. {r['article_id']}] RRF={r['hybrid_score']:.4f} | {r['text'][:50]}...")

🧪 TEST COMPARATIF pour : 'ما هي شروط الحصول على رخصة السياقة'

📊 TF-IDF Top-3 :
  [Art. 239] Score=0.3241 | المادة 239 من هذا القانون، هياات او ماسسات الدولة،...
  [Art. 11] Score=0.2785 | المادة 11 ال يجوز الي كان ان يتقدم الجتياز امتحان ...
  [Art. 3] Score=0.2642 | المادة 3 يجب علي السااقين الحاصلين علي رخصة سياقة ...

🧠 Sémantique Top-3 :
  [Art. 26] Score=0.7976 | المادة 26 يجب علي صاحب رخصة السياقة، الذي فقد خالل...
  [Art. 309] Score=0.7774 | المادة 309 يجب علي االشخاص الحاصلين علي رخصة السيا...
  [Art. 251] Score=0.7604 | المادة 251 ادناه ؛ 4 ان يكون حاصال علي : رخصة السي...

🔀 Hybride (RRF) Top-3 :
  [Art. 239] RRF=0.0164 | المادة 239 من هذا القانون، هياات او ماسسات الدولة،...
  [Art. 26] RRF=0.0164 | المادة 26 يجب علي صاحب رخصة السياقة، الذي فقد خالل...
  [Art. 11] RRF=0.0161 | المادة 11 ال يجوز الي كان ان يتقدم الجتياز امتحان ...


---
## 3. 🤖 LLM Simulé + Prompt Engineering

### Rôle du générateur dans le RAG

Le générateur prend en entrée :
1. Les documents récupérés par le retriever (contexte)
2. La question de l'utilisateur

Et produit une réponse cohérente, ancrée dans les sources.

### Prompt Engineering

La construction du prompt est cruciale : elle structure comment le LLM doit utiliser le contexte pour répondre. Un bon prompt pour du RAG juridique :
- Fixe le rôle du système (assistant juridique)
- Fournit le contexte documentaire
- Demande de citer les sources
- Interdit les réponses non fondées

In [11]:
# ─────────────────────────────────────────────
# 3.1 Construction du Prompt
# ─────────────────────────────────────────────

def build_prompt(question: str, retrieved_docs: list) -> str:
    """
    Construit le prompt structuré pour le LLM.

    Le prompt suit le format RAG standard :
    [SYSTEM] + [CONTEXT] + [QUESTION] + [INSTRUCTION DE RÉPONSE]

    Args:
        question: Question de l'utilisateur
        retrieved_docs: Documents récupérés par le retriever

    Returns:
        Prompt complet prêt à être envoyé au LLM
    """
    # Formatage du contexte documentaire
    context_parts = []
    for i, doc in enumerate(retrieved_docs, 1):
        article_id = doc.get('article_id', '?')
        text = doc.get('text', '')
        mots_cles = doc.get('mots_cles', '')
        amende = doc.get('amende_fixe', None)
        points = doc.get('points_retrait', None)

        doc_str = f"[المستند {i} - المادة {article_id}]\n{text}"
        if mots_cles:
            doc_str += f"\n(الكلمات المفتاحية: {mots_cles})"
        if amende and pd.notna(amende) and amende > 0:
            doc_str += f"\n(الغرامة: {amende} درهم)"
        if points and pd.notna(points) and points > 0:
            doc_str += f"\n(سحب النقاط: {int(points)} نقطة)"

        context_parts.append(doc_str)

    context = "\n\n".join(context_parts)

    # Construction du prompt complet
    prompt = f"""أنت مساعد قانوني متخصص في قانون المرور المغربي.
مهمتك هي الإجابة على الأسئلة بناءً حصرياً على المستندات القانونية المقدمة.
لا تُجب على أي سؤال خارج نطاق هذه المستندات.
استشهد دائماً بأرقام المواد في إجابتك.

═══════════════════════════════════════
السياق القانوني (المستندات المسترجعة):
═══════════════════════════════════════
{context}

═══════════════════════════════════════
السؤال:
═══════════════════════════════════════
{question}

═══════════════════════════════════════
الإجابة (بناءً على المستندات أعلاه فقط):
═══════════════════════════════════════"""

    return prompt


# Démonstration du prompt
demo_question = "ما هي شروط الحصول على رخصة السياقة؟"
demo_docs = get_relevant_documents(demo_question, top_k=3)
demo_prompt = build_prompt(demo_question, demo_docs)

print("📋 EXEMPLE DE PROMPT CONSTRUIT :")
print("=" * 70)
print(demo_prompt)
print("=" * 70)

📋 EXEMPLE DE PROMPT CONSTRUIT :
أنت مساعد قانوني متخصص في قانون المرور المغربي.
مهمتك هي الإجابة على الأسئلة بناءً حصرياً على المستندات القانونية المقدمة.
لا تُجب على أي سؤال خارج نطاق هذه المستندات.
استشهد دائماً بأرقام المواد في إجابتك.

═══════════════════════════════════════
السياق القانوني (المستندات المسترجعة):
═══════════════════════════════════════
[المستند 1 - المادة 239]
المادة 239 من هذا القانون، هياات او ماسسات الدولة، التي تلقن تعليما يف ضي الي الحصول علي بطاقة سااق منهي او علي رخصة مدرب لتعليم السياق ة او منشط دورات التربية علي السالمة الطرقية او رخصة السياقة.
(الكلمات المفتاحية: permis)

[المستند 2 - المادة 26]
المادة 26 يجب علي صاحب رخصة السياقة، الذي فقد خالل الفترة االختبارية اكثر من ثلثي النقط المخصصة للرخصة المذكورة، ان يخضع لدورة في التربية علي السالمة الطرقية تحدد كيفياتها من قبل االدارة.
(الكلمات المفتاحية: permis)

[المستند 3 - المادة 11]
المادة 11 ال يجوز الي كان ان يتقدم الجتياز امتحان الحصول علي رخصة سياقة احدي اصناف المركبات، اذا لم يكن مستوفيا للشروط التالي

In [12]:
# ─────────────────────────────────────────────
# 3.2 LLM Simulé
# ─────────────────────────────────────────────
# NOTE : Pas d'API externe. Le LLM simulé extrait et reformule
# intelligemment les informations des documents récupérés.
# Il cite les articles, structure la réponse, et résume le contenu.

class SimulatedLLM:
    """
    LLM Simulé pour le système RAG juridique.

    Ce générateur simule le comportement d'un LLM en :
    1. Analysant les documents récupérés
    2. Extrayant les informations pertinentes
    3. Structurant une réponse cohérente avec citations
    4. Ajoutant des informations sur les amendes et points si disponibles

    ⚠️ Différence avec un vrai LLM :
    Un vrai LLM (GPT-4, Mistral, Qwen) génère du texte de manière
    autorégressive avec compréhension profonde du langage.
    Ce simulateur extrait et reformule mécaniquement.
    """

    def __init__(self, max_doc_chars: int = 300):
        self.max_doc_chars = max_doc_chars

    def _format_sanctions(self, docs: list) -> str:
        """Extrait et formate les informations de sanctions des documents."""
        sanctions = []
        for doc in docs:
            article_id = doc.get('article_id', '?')
            amende = doc.get('amende_fixe', None)
            points = doc.get('points_retrait', None)
            has_sanction = doc.get('has_sanction', False)

            if has_sanction or (amende and pd.notna(amende) and amende > 0):
                s = f"  📌 المادة {article_id}:"
                if amende and pd.notna(amende) and amende > 0:
                    s += f" غرامة {amende} درهم"
                if points and pd.notna(points) and points > 0:
                    s += f" + سحب {int(points)} نقطة"
                sanctions.append(s)

        return "\n".join(sanctions) if sanctions else ""

    def _extract_key_info(self, text: str, max_chars: int) -> str:
        """Extrait les informations clés d'un texte."""
        if len(text) <= max_chars:
            return text
        # Garder la première phrase complète + résumé
        sentences = re.split(r'(?<=[.،؛])\s+', text)
        result = ""
        for sent in sentences:
            if len(result) + len(sent) < max_chars:
                result += sent + " "
            else:
                break
        return result.strip() + "..."

    def generate(self, question: str, retrieved_docs: list) -> dict:
        """
        Génère une réponse structurée basée sur les documents récupérés.

        Args:
            question: Question de l'utilisateur
            retrieved_docs: Documents récupérés par le retriever

        Returns:
            Dict contenant la réponse, les références, et le prompt utilisé
        """
        if not retrieved_docs:
            return {
                'answer': 'لم يتم العثور على وثائق ذات صلة بسؤالك في قاعدة البيانات القانونية.',
                'references': [],
                'prompt': build_prompt(question, []),
            }

        # Construire le prompt (documenté)
        prompt = build_prompt(question, retrieved_docs)

        # Analyse des documents et construction de la réponse
        answer_parts = []

        # En-tête de la réponse
        answer_parts.append("بناءً على نصوص قانون المرور المغربي، إليك المعلومات المتعلقة بسؤالك:\n")

        # Corps de la réponse : extrait des articles pertinents
        answer_parts.append("📖 المواد القانونية ذات الصلة:")
        references = []

        for i, doc in enumerate(retrieved_docs, 1):
            article_id = doc.get('article_id', '?')
            text = doc.get('text', '')
            key_info = self._extract_key_info(text, self.max_doc_chars)

            answer_parts.append(f"\n{i}. المادة {article_id}:")
            answer_parts.append(f"   {key_info}")
            references.append(f"المادة {article_id}")

        # Section sanctions si disponibles
        sanctions_info = self._format_sanctions(retrieved_docs)
        if sanctions_info:
            answer_parts.append("\n\n⚖️ العقوبات المحتملة:")
            answer_parts.append(sanctions_info)

        # Pied de page avec références
        answer_parts.append(f"\n\n📚 المراجع: {' | '.join(references)}")
        answer_parts.append("\n⚠️ تنبيه: هذه المعلومات مستخرجة حصرياً من قاعدة بيانات قانون المرور المغربي.")

        return {
            'answer': '\n'.join(answer_parts),
            'references': references,
            'prompt': prompt,
            'num_docs_used': len(retrieved_docs),
        }


# Instantiation du LLM simulé
simulated_llm = SimulatedLLM(max_doc_chars=250)

# Test de génération
test_q = "ما هي عقوبة السياقة تحت تأثير الكحول؟"
test_docs = get_relevant_documents(test_q, top_k=4)
test_response = simulated_llm.generate(test_q, test_docs)

print("🤖 RÉPONSE DU LLM SIMULÉ :")
print("=" * 70)
print(f"Question : {test_q}")
print("-" * 70)
print(test_response['answer'])
print("=" * 70)

🤖 RÉPONSE DU LLM SIMULÉ :
Question : ما هي عقوبة السياقة تحت تأثير الكحول؟
----------------------------------------------------------------------
بناءً على نصوص قانون المرور المغربي، إليك المعلومات المتعلقة بسؤالك:

📖 المواد القانونية ذات الصلة:

1. المادة 207:
   المادة 207 ادناه او للتحققات 6 او الختبارات الكشف المنصوص عليها في المادتين 208 و 213 ادناه. 08 سياقة مركبة تحت تاثير االدوية التي تحظر السياقة بعد تناولها. 2

2. المادة 183:
   المادة 183 بتاريخ 7 ذي القعدة 1437 11 اغسطس 2016 ، ص: 5874 يعاقب بالحبس من ستة 6 اشهر الي سنة واحدة وبغرامة من خمسة ااالف 5.000 الي عشرة

3. المادة 209:
   المادة 209 تنجز التحققات الهادفة الي اثبات الحالة الكحولية عن طريق التحاليل والفحوصات الطبية السريرية والبيولوجية او بواسطة جهاز يمكن من تحديد تركز الكحول من خالل تحليل الهواا

4. المادة 152:
   المادة 152 يعاقب بغرامة من الفي 2.000 درهم الي ثمانية ااالف 8.000 درهم، كل شخص صدر في حقه مقرر قضااي حااز لقوة ال ضيا المق ضي به او قرار اداري بتوقيف رخصة السياقة او بسحبها او بالغااها:


⚖️ العقوبات المحتم

---
## 4. 🔗 Pipeline RAG Complet

Le pipeline intègre tous les composants en un flux cohérent :
1. Réception de la question
2. Détection hors domaine
3. Récupération des documents pertinents
4. Construction du prompt
5. Génération de la réponse

In [13]:
# ─────────────────────────────────────────────
# 4.1 Détection de questions hors domaine
# ─────────────────────────────────────────────

# Mots-clés liés au domaine (Code de la route marocain)
DOMAIN_KEYWORDS_AR = [
    'رخصة', 'سياقة', 'مركبة', 'طريق', 'سرعة', 'غرامة', 'مخالفة',
    'نقطة', 'سائق', 'حادثة', 'مرور', 'إشارة', 'وقوف', 'تجاوز',
    'كحول', 'حزام', 'خوذة', 'دراجة', 'شاحنة', 'حافلة', 'مدرسة',
    'امتحان', 'تامين', 'تسجيل', 'لوحة', 'ضباب', 'إضاءة', 'فرامل',
    'أولوية', 'دوار', 'نفق', 'جسر', 'محور', 'تقاطع', 'منعطف',
]

DOMAIN_KEYWORDS_FR = [
    'permis', 'conduire', 'vehicule', 'route', 'vitesse', 'amende',
    'infraction', 'point', 'conducteur', 'accident', 'circulation',
    'signal', 'parking', 'depassement', 'alcool', 'ceinture', 'casque',
    'moto', 'camion', 'bus', 'examen', 'assurance', 'immatriculation',
    'feux', 'frein', 'priorite', 'rond-point', 'tunnel',
]

# Mots-clés hors domaine
OUT_OF_DOMAIN_KEYWORDS = [
    'طبخ', 'رياضة', 'موسيقى', 'أفلام', 'سياسة', 'تاريخ', 'علوم',
    'رياضيات', 'طب', 'اقتصاد', 'مناخ', 'جغرافيا', 'فلسفة',
    'cuisine', 'sport', 'musique', 'film', 'politique', 'histoire',
    'science', 'mathématique', 'médecine', 'économie', 'météo',
    'weather', 'cooking', 'football', 'basketball',
]

SIMILARITY_THRESHOLD = 0.15  # Seuil minimal de similarité sémantique


def detect_out_of_domain(question: str, top_k: int = 3) -> dict:
    """
    Détecte si une question est hors du domaine du Code de la Route.

    Méthode hybride :
    1. Règles par mots-clés (domaine + hors domaine)
    2. Seuil de similarité sémantique avec le corpus

    Args:
        question: Question de l'utilisateur
        top_k: Nombre de documents à vérifier pour le seuil

    Returns:
        Dict avec is_out_of_domain (bool) et reason (str)
    """
    question_lower = question.lower()

    # Règle 1 : Question trop courte
    if len(question.strip()) < 5:
        return {'is_out_of_domain': True, 'reason': 'السؤال قصير جداً', 'confidence': 1.0}

    # Règle 2 : Présence de mots-clés hors domaine explicites
    for kw in OUT_OF_DOMAIN_KEYWORDS:
        if kw in question_lower:
            return {
                'is_out_of_domain': True,
                'reason': f'السؤال يتعلق بموضوع خارج نطاق قانون المرور: "{kw}"',
                'confidence': 0.95
            }

    # Règle 3 : Présence de mots-clés du domaine → IN DOMAIN
    all_domain_kw = DOMAIN_KEYWORDS_AR + DOMAIN_KEYWORDS_FR
    domain_hits = [kw for kw in all_domain_kw if kw in question_lower]
    if domain_hits:
        return {
            'is_out_of_domain': False,
            'reason': f'مطابقة مع كلمات مفتاحية: {domain_hits[:3]}',
            'confidence': 0.9
        }

    # Règle 4 : Seuil de similarité sémantique
    semantic_results = retrieve_semantic(question, top_k=top_k)
    if not semantic_results:
        return {'is_out_of_domain': True, 'reason': 'لا توجد وثائق ذات صلة', 'confidence': 0.8}

    max_similarity = max(r.get('semantic_score', 0) for r in semantic_results)

    if max_similarity < SIMILARITY_THRESHOLD:
        return {
            'is_out_of_domain': True,
            'reason': f'تشابه منخفض مع قاعدة البيانات (أعلى نتيجة: {max_similarity:.3f} < {SIMILARITY_THRESHOLD})',
            'confidence': 0.75
        }

    return {
        'is_out_of_domain': False,
        'reason': f'تشابه كافٍ ({max_similarity:.3f})',
        'confidence': min(max_similarity, 0.99)
    }


# Tests de détection hors domaine
test_questions = [
    ("ما هي شروط الحصول على رخصة السياقة؟", "IN DOMAIN"),
    ("ما هي عقوبة تجاوز السرعة المسموح بها؟", "IN DOMAIN"),
    ("من هو أفضل لاعب كرة قدم في العالم؟", "OUT OF DOMAIN"),
    ("كيف أطبخ الكسكس المغربي؟", "OUT OF DOMAIN"),
    ("ما هي عاصمة فرنسا؟", "OUT OF DOMAIN"),
    ("ما هي نقاط سحب الرخصة عند السياقة في حالة سكر؟", "IN DOMAIN"),
]

print("🧪 TESTS DE DÉTECTION HORS DOMAINE")
print("=" * 80)
for question, expected in test_questions:
    result = detect_out_of_domain(question)
    status = "❌ HORS DOMAINE" if result['is_out_of_domain'] else "✅ IN DOMAIN"
    correct = "✓" if (result['is_out_of_domain'] == (expected == "OUT OF DOMAIN")) else "✗"
    print(f"{correct} {status} [{expected}] | {question[:50]}")
    print(f"   → Raison: {result['reason']}")
    print()

🧪 TESTS DE DÉTECTION HORS DOMAINE
✓ ✅ IN DOMAIN [IN DOMAIN] | ما هي شروط الحصول على رخصة السياقة؟
   → Raison: مطابقة مع كلمات مفتاحية: ['رخصة', 'سياقة']

✓ ✅ IN DOMAIN [IN DOMAIN] | ما هي عقوبة تجاوز السرعة المسموح بها؟
   → Raison: مطابقة مع كلمات مفتاحية: ['سرعة', 'تجاوز']

✗ ✅ IN DOMAIN [OUT OF DOMAIN] | من هو أفضل لاعب كرة قدم في العالم؟
   → Raison: تشابه كافٍ (0.164)

✓ ❌ HORS DOMAINE [OUT OF DOMAIN] | كيف أطبخ الكسكس المغربي؟
   → Raison: السؤال يتعلق بموضوع خارج نطاق قانون المرور: "طبخ"

✗ ✅ IN DOMAIN [OUT OF DOMAIN] | ما هي عاصمة فرنسا؟
   → Raison: تشابه كافٍ (0.258)

✓ ✅ IN DOMAIN [IN DOMAIN] | ما هي نقاط سحب الرخصة عند السياقة في حالة سكر؟
   → Raison: مطابقة مع كلمات مفتاحية: ['رخصة', 'سياقة']



In [14]:
# ─────────────────────────────────────────────
# 4.2 Pipeline RAG Complet
# ─────────────────────────────────────────────

class RAGPipeline:
    """
    Pipeline RAG Complet pour le Code de la Route Marocain.

    Ce pipeline orchestre :
    1. Réception et validation de la question
    2. Détection des questions hors domaine
    3. Récupération hybride des documents pertinents (TF-IDF + FAISS)
    4. Construction du prompt structuré
    5. Génération de la réponse via LLM simulé
    6. Retour de la réponse avec références et métadonnées
    """

    def __init__(self, llm: SimulatedLLM, top_k: int = 5):
        self.llm = llm
        self.top_k = top_k
        self.query_history = []

    def run(self, question: str, verbose: bool = False) -> dict:
        """
        Exécute le pipeline RAG complet.

        Args:
            question: Question en langage naturel
            verbose: Afficher les étapes intermédiaires

        Returns:
            Dict contenant answer, references, docs, metadata
        """
        if verbose:
            print(f"\n{'='*60}")
            print(f"🔍 Question: {question}")
            print(f"{'='*60}")

        # ── Étape 1 : Détection hors domaine ──
        domain_check = detect_out_of_domain(question)
        if verbose:
            print(f"\n[1] Détection domaine : {domain_check}")

        if domain_check['is_out_of_domain']:
            response = {
                'answer': f"عذراً، سؤالك خارج نطاق قانون المرور المغربي.\n\nالسبب: {domain_check['reason']}\n\nيمكنني فقط الإجابة على الأسئلة المتعلقة بقانون المرور (رخص السياقة، المخالفات، الغرامات، قواعد السير، إلخ).",
                'references': [],
                'retrieved_docs': [],
                'is_out_of_domain': True,
                'domain_reason': domain_check['reason'],
                'num_docs_retrieved': 0,
            }
            self.query_history.append({'question': question, 'status': 'out_of_domain'})
            return response

        # ── Étape 2 : Récupération hybride ──
        retrieved_docs = get_relevant_documents(question, top_k=self.top_k)
        if verbose:
            print(f"\n[2] Documents récupérés : {len(retrieved_docs)}")
            for d in retrieved_docs:
                print(f"    [Art. {d['article_id']}] score={d.get('hybrid_score', 0):.4f}")

        # ── Étape 3 : Construction du prompt ──
        prompt = build_prompt(question, retrieved_docs)
        if verbose:
            print(f"\n[3] Prompt construit ({len(prompt)} chars)")

        # ── Étape 4 : Génération de la réponse ──
        llm_output = self.llm.generate(question, retrieved_docs)
        if verbose:
            print(f"\n[4] Réponse générée")

        # Enregistrement dans l'historique
        self.query_history.append({
            'question': question,
            'status': 'answered',
            'num_docs': len(retrieved_docs),
            'references': llm_output['references'],
        })

        return {
            'answer': llm_output['answer'],
            'references': llm_output['references'],
            'retrieved_docs': retrieved_docs,
            'prompt': prompt,
            'is_out_of_domain': False,
            'num_docs_retrieved': len(retrieved_docs),
        }


# Initialisation du pipeline
rag_pipeline = RAGPipeline(llm=simulated_llm, top_k=5)

# ─────── Tests du pipeline ───────
demo_questions = [
    "ما هي شروط الحصول على رخصة السياقة المغربية؟",
    "كيف أطبخ الكسكس المغربي؟",  # Hors domaine
    "ما هي عقوبة تجاوز الحد الأقصى للسرعة في المناطق الحضرية؟",
]

for q in demo_questions:
    result = rag_pipeline.run(q, verbose=True)
    print("\n📤 RÉPONSE FINALE:")
    print(result['answer'])
    print(f"\n📎 Références: {result['references']}")
    print("\n" + "─"*70)


🔍 Question: ما هي شروط الحصول على رخصة السياقة المغربية؟

[1] Détection domaine : {'is_out_of_domain': False, 'reason': "مطابقة مع كلمات مفتاحية: ['رخصة', 'سياقة']", 'confidence': 0.9}

[2] Documents récupérés : 5
    [Art. 3] score=0.0328
    [Art. 119] score=0.0304
    [Art. 239] score=0.0161
    [Art. 11] score=0.0159
    [Art. 2] score=0.0159

[3] Prompt construit (1665 chars)

[4] Réponse générée

📤 RÉPONSE FINALE:
بناءً على نصوص قانون المرور المغربي، إليك المعلومات المتعلقة بسؤالك:

📖 المواد القانونية ذات الصلة:

1. المادة 3:
   المادة 3 يجب علي السااقين الحاصلين علي رخصة سياقة مسلمة بالخارج، بعد انصرام المدة المشار اليها في المادة السابقة، ان يتقدموا المتحانات الحصول علي رخصة السياقة المغربية، او ان يطلبوا تبديل

2. المادة 119:
   المادة 119 كل مالك مركبة اجنبية، التتوفر علي رقم تسجيل مغربي، يقوم بعملية النقل بين نقطتين داخل التراب المغربي، دون ترخيص خاص مسلم من قبل مصالح السلطة الحكومية المكلفة بالنقل، يعاقب

3. المادة 239:
   المادة 239 من هذا القانون، هياات او ماسسات الدولة،

---
## 5. 📊 Comparaison Théorique de 3 LLMs

Cette section présente une analyse comparative des trois modèles de langage cités dans le sujet, dans le contexte d'un système RAG juridique.

> ⚠️ **Note** : Aucune API n'est utilisée. Cette section est purement analytique et théorique.

### 5.1 GPT-4 (OpenAI)

**Forces :**
- Compréhension contextuelle exceptionnelle, idéale pour suivre des raisonnements juridiques complexes
- Support multilingue robuste incluant l'arabe dialectal et classique
- Fenêtre de contexte étendue (128K tokens) permettant d'injecter de nombreux documents
- Fine-tuning disponible pour spécialisation juridique

**Faiblesses :**
- Coût élevé par requête (critique pour des volumes importants de questions)
- Dépendance à une API externe (confidentialité des données juridiques sensibles)
- Latence variable selon la charge du serveur
- Boîte noire : impossible d'auditer le processus de génération

**Risque d'hallucination pour le RAG juridique :**
- Moyen. GPT-4 peut parfois "compléter" des articles de loi avec des informations plausibles mais incorrectes
- Risque mitigé avec un prompt strict de type RAG ("réponds uniquement selon le contexte")

**Adéquation RAG juridique :** ⭐⭐⭐⭐⭐ (excellent avec un bon prompt engineering)

---

### 5.2 Mistral 7B / Mistral Large (Mistral AI)

**Forces :**
- Open-source : déployable localement (protection des données juridiques)
- Excellentes performances pour sa taille, surtout sur le français
- Architecture Sliding Window Attention : efficace sur de longs contextes
- Mistral Large rivalise avec GPT-4 sur de nombreux benchmarks

**Faiblesses :**
- Support de l'arabe inférieur à GPT-4 (entraîné majoritairement sur données anglaises/françaises)
- Le modèle 7B nécessite quantification pour tourner sur GPU standard
- Moins de ressources de fine-tuning pour le domaine juridique marocain

**Risque d'hallucination pour le RAG juridique :**
- Moyen-faible si les instructions RAG sont claires
- Risque plus élevé sur les textes arabes (moins représentés dans l'entraînement)

**Adéquation RAG juridique :** ⭐⭐⭐⭐ (très bon, surtout pour textes en français)

---

### 5.3 Qwen2.5 (Alibaba)

**Forces :**
- **Excellent support de l'arabe** : entraîné sur un large corpus multilingue incluant beaucoup de données arabes
- Open-source : déployable localement en toute confidentialité
- Versions multiples (0.5B à 72B) : scalable selon les ressources
- Très bonnes performances sur les benchmarks de raisonnement

**Faiblesses :**
- Moins connu et moins documenté que GPT ou Mistral
- Communauté plus petite, moins d'exemples d'usage RAG
- Le fine-tuning sur données juridiques marocaines reste peu exploré

**Risque d'hallucination pour le RAG juridique :**
- Faible à moyen : Qwen respecte généralement bien les contraintes contextuelles
- **Recommandé pour les textes arabes** grâce à sa maîtrise de la langue

**Adéquation RAG juridique (arabe) :** ⭐⭐⭐⭐⭐ (meilleur choix pour l'arabe)

---

### Tableau Comparatif

In [15]:
# Tableau de comparaison des LLMs
comparison_data = {
    'Critère': [
        'Support Arabe', 'Support Français', 'Open Source',
        'Déploiement Local', 'Coût API', 'Fenêtre Contexte',
        'Hallucination (RAG)', 'Compréhension Juridique',
        'Performance Globale', 'Recommandé pour ce projet'
    ],
    'GPT-4 (OpenAI)': [
        '⭐⭐⭐⭐', '⭐⭐⭐⭐⭐', '❌',
        '❌ (API cloud)', '💰💰💰', '128K tokens',
        'Moyen', '⭐⭐⭐⭐⭐',
        '⭐⭐⭐⭐⭐', '✅ (si budget disponible)'
    ],
    'Mistral 7B (Mistral AI)': [
        '⭐⭐', '⭐⭐⭐⭐⭐', '✅',
        '✅ GPU requis', '💰 (faible)', '32K tokens',
        'Moyen-faible', '⭐⭐⭐⭐',
        '⭐⭐⭐⭐', '✅ (pour textes FR)'
    ],
    'Qwen2.5 (Alibaba)': [
        '⭐⭐⭐⭐⭐', '⭐⭐⭐⭐', '✅',
        '✅ GPU requis', '💰 (faible)', '128K tokens',
        'Faible', '⭐⭐⭐⭐',
        '⭐⭐⭐⭐⭐', '✅✅ (MEILLEUR pour AR)'
    ]
}

comparison_df = pd.DataFrame(comparison_data)
comparison_df = comparison_df.set_index('Critère')
print("📊 TABLEAU COMPARATIF DES LLMs")
print("=" * 80)
print(comparison_df.to_string())
print("\n💡 Recommandation : Qwen2.5 est le meilleur choix pour ce projet (textes arabes, open-source, local)")

📊 TABLEAU COMPARATIF DES LLMs
                                     GPT-4 (OpenAI) Mistral 7B (Mistral AI)      Qwen2.5 (Alibaba)
Critère                                                                                           
Support Arabe                                  ⭐⭐⭐⭐                      ⭐⭐                  ⭐⭐⭐⭐⭐
Support Français                              ⭐⭐⭐⭐⭐                   ⭐⭐⭐⭐⭐                   ⭐⭐⭐⭐
Open Source                                       ❌                       ✅                      ✅
Déploiement Local                     ❌ (API cloud)            ✅ GPU requis           ✅ GPU requis
Coût API                                        💰💰💰              💰 (faible)             💰 (faible)
Fenêtre Contexte                        128K tokens              32K tokens            128K tokens
Hallucination (RAG)                           Moyen            Moyen-faible                 Faible
Compréhension Juridique                       ⭐⭐⭐⭐⭐                    ⭐⭐⭐⭐    

---
## 6. 📈 Évaluation des Performances

L'évaluation d'un système RAG comprend deux niveaux :
1. **Évaluation du Retriever** : La précision et le rappel mesurent si les bons documents sont récupérés
2. **Évaluation du Générateur** : ROUGE score mesure la similarité entre la réponse générée et une réponse de référence

In [17]:
# ─────────────────────────────────────────────
# 6.1 Jeu de données d'évaluation
# ─────────────────────────────────────────────
# Questions de test avec les articles attendus

eval_dataset = [
    {
        'question': 'ما هي شروط الحصول على رخصة السياقة؟',
        'expected_article_ids': [10, 11, 12, 23],
        'expected_answer_keywords': ['امتحان', 'رخصة', 'شروط', 'اختبار'],
    },
    {
        'question': 'كم عدد النقاط المخصصة لرخصة السياقة الجديدة؟',
        'expected_article_ids': [22, 23, 27],
        'expected_answer_keywords': ['نقطة', 'رصيد', 'رخصة'],
    },
    {
        'question': 'ما هي مدة الفحص الطبي لرخصة السياقة؟',
        'expected_article_ids': [14, 12, 15, 16],
        'expected_answer_keywords': ['فحص', 'طبي', 'سنة'],
    },
    {
        'question': 'ما هي عواقب فقدان جميع نقاط رخصة السياقة؟',
        'expected_article_ids': [24, 31, 32],
        'expected_answer_keywords': ['إلغاء', 'رخصة', 'نقطة', 'فقدان'],
    },
    {
        'question': 'هل يمكن للمغاربة المقيمين بالخارج السياقة في المغرب؟',
        'expected_article_ids': [2, 3, 4],
        'expected_answer_keywords': ['مغاربة', 'خارج', 'رخصة', 'وطني'],
    },
]

print(f"📊 Jeu d'évaluation : {len(eval_dataset)} questions de test")

📊 Jeu d'évaluation : 5 questions de test


In [18]:
# ─────────────────────────────────────────────
# 6.2 Calcul de Précision et Rappel du Retriever
# ─────────────────────────────────────────────

def compute_precision_recall(retrieved_ids: list, expected_ids: list) -> dict:
    """
    Calcule la précision et le rappel pour un ensemble de documents récupérés.

    Précision = |Récupérés ∩ Attendus| / |Récupérés|
    Rappel    = |Récupérés ∩ Attendus| / |Attendus|
    F1        = 2 × (Précision × Rappel) / (Précision + Rappel)

    Args:
        retrieved_ids: Liste des IDs d'articles récupérés
        expected_ids: Liste des IDs d'articles attendus (ground truth)

    Returns:
        Dict avec precision, recall, f1
    """
    retrieved_set = set(retrieved_ids)
    expected_set = set(expected_ids)

    true_positives = retrieved_set & expected_set

    precision = len(true_positives) / len(retrieved_set) if retrieved_set else 0
    recall = len(true_positives) / len(expected_set) if expected_set else 0
    f1 = (2 * precision * recall / (precision + recall)
          if (precision + recall) > 0 else 0)

    return {
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'true_positives': len(true_positives),
        'retrieved_correct': list(true_positives),
    }


def evaluate_retriever(eval_data: list, top_k: int = 5) -> pd.DataFrame:
    """
    Évalue le retriever hybride sur l'ensemble de questions de test.

    Pour chaque question, compare les articles récupérés aux articles attendus.

    Args:
        eval_data: Jeu d'évaluation
        top_k: Nombre de documents à récupérer

    Returns:
        DataFrame avec les métriques par question
    """
    results = []

    for item in eval_data:
        question = item['question']
        expected_ids = item['expected_article_ids']

        # Récupération hybride
        retrieved = get_relevant_documents(question, top_k=top_k)
        retrieved_ids = [int(d['article_id']) for d in retrieved]

        # Calcul des métriques
        metrics = compute_precision_recall(retrieved_ids, expected_ids)

        results.append({
            'question': question[:50] + '...' if len(question) > 50 else question,
            'expected_ids': expected_ids,
            'retrieved_ids': retrieved_ids,
            'precision': round(metrics['precision'], 3),
            'recall': round(metrics['recall'], 3),
            'f1': round(metrics['f1'], 3),
            'true_positives': metrics['true_positives'],
        })

    return pd.DataFrame(results)


# Évaluation du retriever avec différents top_k
print("📊 ÉVALUATION DU RETRIEVER HYBRIDE")
print("=" * 80)

for top_k in [3, 5, 10]:
    eval_df = evaluate_retriever(eval_dataset, top_k=top_k)
    avg_precision = eval_df['precision'].mean()
    avg_recall = eval_df['recall'].mean()
    avg_f1 = eval_df['f1'].mean()
    print(f"\n📌 Top-{top_k} :")
    print(f"   Précision moyenne  : {avg_precision:.3f}")
    print(f"   Rappel moyen       : {avg_recall:.3f}")
    print(f"   F1-Score moyen     : {avg_f1:.3f}")

print("\n\n📋 Détail par question (Top-5) :")
eval_df_5 = evaluate_retriever(eval_dataset, top_k=5)
print(eval_df_5[['question', 'precision', 'recall', 'f1', 'true_positives']].to_string(index=False))

📊 ÉVALUATION DU RETRIEVER HYBRIDE

📌 Top-3 :
   Précision moyenne  : 0.333
   Rappel moyen       : 0.300
   F1-Score moyen     : 0.314

📌 Top-5 :
   Précision moyenne  : 0.187
   Rappel moyen       : 0.250
   F1-Score moyen     : 0.207

📌 Top-10 :
   Précision moyenne  : 0.150
   Rappel moyen       : 0.417
   F1-Score moyen     : 0.218


📋 Détail par question (Top-5) :
                                             question  precision  recall    f1  true_positives
                  ما هي شروط الحصول على رخصة السياقة؟      0.000   0.000 0.000               0
         كم عدد النقاط المخصصة لرخصة السياقة الجديدة؟      0.000   0.000 0.000               0
                 ما هي مدة الفحص الطبي لرخصة السياقة؟      0.333   0.250 0.286               1
            ما هي عواقب فقدان جميع نقاط رخصة السياقة؟      0.200   0.333 0.250               1
هل يمكن للمغاربة المقيمين بالخارج السياقة في المغر...      0.400   0.667 0.500               2


In [19]:
# ─────────────────────────────────────────────
# 6.3 Score ROUGE (Évaluation de la réponse)
# ─────────────────────────────────────────────
try:
    from rouge_score import rouge_scorer
    ROUGE_AVAILABLE = True
except ImportError:
    ROUGE_AVAILABLE = False
    print("⚠️ rouge_score non disponible. Calcul ROUGE désactivé.")

# Réponses de référence (ground truth simplifiées)
reference_answers = [
    "يجب اجتياز امتحان نظري وعملي والخضوع لفحص طبي للحصول على رخصة السياقة",
    "يخصص لرخصة السياقة رصيد من النقاط يتم تخفيضه عند ارتكاب المخالفات",
    "يجب على الحاصلين على رخصة السياقة الخضوع لفحص طبي كل عشر سنوات",
    "تلغى رخصة السياقة عند فقدان جميع النقاط وتسلم رسالة للمعني بالأمر",
    "يحق للمغاربة المقيمين بالخارج السياقة داخل التراب الوطني لمدة محددة",
]

if ROUGE_AVAILABLE:
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=False)

    def compute_rouge(generated: str, reference: str) -> dict:
        """Calcule les scores ROUGE entre réponse générée et référence."""
        scores = scorer.score(reference, generated)
        return {
            'rouge1_f': scores['rouge1'].fmeasure,
            'rouge2_f': scores['rouge2'].fmeasure,
            'rougeL_f': scores['rougeL'].fmeasure,
        }

    print("📊 ÉVALUATION ROUGE DU GÉNÉRATEUR")
    print("=" * 80)

    rouge_results = []
    for i, (item, reference) in enumerate(zip(eval_dataset, reference_answers)):
        result = rag_pipeline.run(item['question'])
        generated = result['answer']
        rouge = compute_rouge(generated, reference)
        rouge_results.append(rouge)

        print(f"Q{i+1}: {item['question'][:50]}...")
        print(f"     ROUGE-1: {rouge['rouge1_f']:.3f} | ROUGE-2: {rouge['rouge2_f']:.3f} | ROUGE-L: {rouge['rougeL_f']:.3f}")

    avg_rouge = pd.DataFrame(rouge_results).mean()
    print(f"\n📊 Moyennes ROUGE :")
    print(f"   ROUGE-1 : {avg_rouge['rouge1_f']:.3f}")
    print(f"   ROUGE-2 : {avg_rouge['rouge2_f']:.3f}")
    print(f"   ROUGE-L : {avg_rouge['rougeL_f']:.3f}")
else:
    print("Calcul ROUGE ignoré (bibliothèque non disponible).")
    print("⚠️ Note : Les scores ROUGE mesure la similarité de surface entre généré et référence.")
    print("Pour un vrai LLM (Mistral/Qwen/GPT), les scores ROUGE seraient plus élevés")
    print("car la génération serait plus fluide et variée tout en restant fidèle au contexte.")

📊 ÉVALUATION ROUGE DU GÉNÉRATEUR
Q1: ما هي شروط الحصول على رخصة السياقة؟...
     ROUGE-1: 0.000 | ROUGE-2: 0.000 | ROUGE-L: 0.000
Q2: كم عدد النقاط المخصصة لرخصة السياقة الجديدة؟...
     ROUGE-1: 0.000 | ROUGE-2: 0.000 | ROUGE-L: 0.000
Q3: ما هي مدة الفحص الطبي لرخصة السياقة؟...
     ROUGE-1: 0.000 | ROUGE-2: 0.000 | ROUGE-L: 0.000
Q4: ما هي عواقب فقدان جميع نقاط رخصة السياقة؟...
     ROUGE-1: 0.000 | ROUGE-2: 0.000 | ROUGE-L: 0.000
Q5: هل يمكن للمغاربة المقيمين بالخارج السياقة في المغر...
     ROUGE-1: 0.000 | ROUGE-2: 0.000 | ROUGE-L: 0.000

📊 Moyennes ROUGE :
   ROUGE-1 : 0.000
   ROUGE-2 : 0.000
   ROUGE-L : 0.000


In [20]:
# ─────────────────────────────────────────────
# 6.4 Comparaison TF-IDF vs Sémantique vs Hybride
# ─────────────────────────────────────────────
print("📊 COMPARAISON DES STRATÉGIES DE RETRIEVAL")
print("=" * 80)

strategies = {
    'TF-IDF': lambda q, k: [int(d['article_id']) for d in retrieve_tfidf(q, top_k=k)],
    'Sémantique (FAISS)': lambda q, k: [int(d['article_id']) for d in retrieve_semantic(q, top_k=k)],
    'Hybride (RRF)': lambda q, k: [int(d['article_id']) for d in get_relevant_documents(q, top_k=k)],
}

comparison_results = {}
top_k = 5

for strategy_name, retriever_fn in strategies.items():
    precisions, recalls, f1s = [], [], []
    for item in eval_dataset:
        retrieved_ids = retriever_fn(item['question'], top_k)
        metrics = compute_precision_recall(retrieved_ids, item['expected_article_ids'])
        precisions.append(metrics['precision'])
        recalls.append(metrics['recall'])
        f1s.append(metrics['f1'])

    comparison_results[strategy_name] = {
        'Précision': np.mean(precisions),
        'Rappel': np.mean(recalls),
        'F1': np.mean(f1s),
    }

comp_df = pd.DataFrame(comparison_results).T.round(3)
print(comp_df.to_string())
print("\n💡 Le retriever hybride (RRF) combine les forces des deux approches.")

📊 COMPARAISON DES STRATÉGIES DE RETRIEVAL
                    Précision  Rappel     F1
TF-IDF                  0.300   0.417  0.344
Sémantique (FAISS)      0.160   0.233  0.189
Hybride (RRF)           0.187   0.250  0.207

💡 Le retriever hybride (RRF) combine les forces des deux approches.


---
## 7. 🚫 Détection de Questions Hors Domaine (Section Dédiée)

In [21]:
# ─────────────────────────────────────────────
# 7.1 Tests exhaustifs de détection hors domaine
# ─────────────────────────────────────────────

extensive_test_cases = [
    # Questions du domaine
    {"q": "ما هي عقوبة السياقة بدون رخصة؟", "expected": False},
    {"q": "كيف يتم تجديد رخصة السياقة؟", "expected": False},
    {"q": "ما هو الحد الأقصى للسرعة في الطريق السريع؟", "expected": False},
    {"q": "ما هي عقوبة السياقة في حالة السكر؟", "expected": False},
    {"q": "هل يجب ارتداء حزام الأمان؟", "expected": False},
    {"q": "ما هي الأصناف المختلفة لرخصة السياقة المغربية؟", "expected": False},
    # Questions hors domaine
    {"q": "من هو رئيس الحكومة المغربية؟", "expected": True},
    {"q": "ما هي أفضل وصفة للكسكس؟", "expected": True},
    {"q": "كيف أتعلم البرمجة بلغة Python؟", "expected": True},
    {"q": "ما هي درجة الحرارة في مراكش اليوم؟", "expected": True},
    {"q": "من هو أفضل لاعب كرة قدم في التاريخ؟", "expected": True},
    {"q": "كم عدد سكان المغرب؟", "expected": True},
]

print("🧪 ÉVALUATION DU DÉTECTEUR HORS DOMAINE")
print("=" * 80)

correct = 0
results_detail = []

for case in extensive_test_cases:
    detection = detect_out_of_domain(case['q'])
    predicted_out = detection['is_out_of_domain']
    expected_out = case['expected']
    is_correct = (predicted_out == expected_out)
    if is_correct:
        correct += 1

    status_icon = "✅" if is_correct else "❌"
    domain_icon = "🚫" if predicted_out else "✅"
    results_detail.append({
        'Question': case['q'][:45] + '...' if len(case['q']) > 45 else case['q'],
        'Attendu': 'Hors domaine' if expected_out else 'In domain',
        'Prédit': 'Hors domaine' if predicted_out else 'In domain',
        'Correct': status_icon,
    })

results_table = pd.DataFrame(results_detail)
print(results_table.to_string(index=False))

accuracy = correct / len(extensive_test_cases)
print(f"\n📊 Précision de détection hors domaine : {correct}/{len(extensive_test_cases)} = {accuracy:.1%}")

🧪 ÉVALUATION DU DÉTECTEUR HORS DOMAINE
                                        Question      Attendu       Prédit Correct
                  ما هي عقوبة السياقة بدون رخصة؟    In domain    In domain       ✅
                     كيف يتم تجديد رخصة السياقة؟    In domain    In domain       ✅
      ما هو الحد الأقصى للسرعة في الطريق السريع؟    In domain    In domain       ✅
              ما هي عقوبة السياقة في حالة السكر؟    In domain    In domain       ✅
                      هل يجب ارتداء حزام الأمان؟    In domain    In domain       ✅
ما هي الأصناف المختلفة لرخصة السياقة المغربية...    In domain    In domain       ✅
                    من هو رئيس الحكومة المغربية؟ Hors domaine    In domain       ❌
                         ما هي أفضل وصفة للكسكس؟ Hors domaine    In domain       ❌
                  كيف أتعلم البرمجة بلغة Python؟ Hors domaine    In domain       ❌
              ما هي درجة الحرارة في مراكش اليوم؟ Hors domaine    In domain       ❌
             من هو أفضل لاعب كرة قدم في التاريخ؟

---
## 8. 🖥️ Interface Utilisateur

Interface web interactive permettant d'interroger le système RAG via Gradio.

In [22]:
# ─────────────────────────────────────────────
# 8.1 Interface Gradio
# ─────────────────────────────────────────────
import gradio as gr

# ── Fonctions d'interface ──
def format_retrieved_docs(docs: list) -> str:
    """Formate les documents récupérés pour l'affichage."""
    if not docs:
        return "Aucun document récupéré."
    lines = []
    for i, doc in enumerate(docs, 1):
        score = doc.get('hybrid_score', doc.get('semantic_score', 0))
        amende = doc.get('amende_fixe', None)
        points = doc.get('points_retrait', None)

        line = f"📄 [{i}] المادة {doc['article_id']} (Score: {score:.4f})"
        if amende and pd.notna(amende) and amende > 0:
            line += f" | 💰 {amende} MAD"
        if points and pd.notna(points) and points > 0:
            line += f" | ⚠️ -{int(points)} pts"
        line += f"\n   {doc['text'][:150]}..."
        lines.append(line)
    return "\n\n".join(lines)


def rag_interface(question: str, top_k: int) -> tuple:
    """
    Fonction principale de l'interface Gradio.
    Exécute le pipeline RAG et retourne les résultats formatés.

    Args:
        question: Question de l'utilisateur
        top_k: Nombre de documents à récupérer

    Returns:
        Tuple (réponse, documents récupérés, statut)
    """
    if not question or not question.strip():
        return "⚠️ الرجاء إدخال سؤال.", "", "⚠️ سؤال فارغ"

    # Mise à jour du top_k
    rag_pipeline.top_k = int(top_k)

    # Exécution du pipeline
    result = rag_pipeline.run(question, verbose=False)

    answer = result['answer']
    docs_display = format_retrieved_docs(result.get('retrieved_docs', []))

    if result.get('is_out_of_domain', False):
        status = "🚫 سؤال خارج نطاق قانون المرور"
    else:
        refs = ', '.join([f"المادة {r.split()[-1]}" for r in result.get('references', [])])
        status = f"✅ تمت الإجابة | المراجع: {refs} | عدد المستندات: {result['num_docs_retrieved']}"

    return answer, docs_display, status


# ── Construction de l'interface ──
SAMPLE_QUESTIONS = [
    "ما هي شروط الحصول على رخصة السياقة المغربية؟",
    "كم عدد النقاط التي يتم سحبها عند تجاوز السرعة؟",
    "ما هي مدة الفترة التجريبية لرخصة السياقة الجديدة؟",
    "ما هي عقوبة السياقة في حالة السكر؟",
    "هل يمكن للمغاربة المقيمين بالخارج السياقة في المغرب؟",
    "كيف أطبخ الكسكس المغربي؟",  # Hors domaine
]

with gr.Blocks(
    title="نظام RAG للقانون المغربي",
    theme=gr.themes.Soft(),
    css=".gradio-container { direction: rtl; font-family: 'Cairo', Arial, sans-serif; }"
) as demo:

    gr.HTML("""
    <div style='text-align:center; padding:20px; background: linear-gradient(135deg, #1a3a5c, #2ecc71); border-radius:12px; margin-bottom:20px'>
        <h1 style='color:white; font-size:28px; margin:0'>🏛️ نظام الاستفسار الذكي عن قانون المرور المغربي</h1>
        <p style='color:#ecf0f1; margin:8px 0 0'>Système RAG | Master IASD 2026 | Université Abdelmalek Essaadi</p>
    </div>
    """)

    with gr.Row():
        with gr.Column(scale=2):
            question_input = gr.Textbox(
                label="❓ سؤالك حول قانون المرور",
                placeholder="مثال: ما هي شروط الحصول على رخصة السياقة؟",
                lines=3,
                rtl=True,
            )
            top_k_slider = gr.Slider(
                minimum=1, maximum=10, value=5, step=1,
                label="عدد المستندات المسترجعة (Top-K)"
            )

            with gr.Row():
                submit_btn = gr.Button("🔍 ابحث", variant="primary", size="lg")
                clear_btn = gr.Button("🗑️ مسح", variant="secondary")

            gr.Examples(
                examples=[[q] for q in SAMPLE_QUESTIONS],
                inputs=question_input,
                label="📝 أسئلة نموذجية",
            )

        with gr.Column(scale=1):
            gr.HTML("""
            <div style='background:#f8f9fa; padding:15px; border-radius:8px; border-right:4px solid #2ecc71'>
                <h3>🏗️ معلومات النظام</h3>
                <p>• <b>المسترجع:</b> هجين (TF-IDF + FAISS)</p>
                <p>• <b>التضمين:</b> multilingual-MiniLM-L12</p>
                <p>• <b>المولد:</b> LLM محاكاة ذكية</p>
                <p>• <b>كشف خارج النطاق:</b> قواعد + عتبة تشابه</p>
            </div>
            """)

    status_output = gr.Textbox(label="📊 الحالة", interactive=False)

    with gr.Tabs():
        with gr.TabItem("💬 الإجابة"):
            answer_output = gr.Textbox(
                label="الإجابة القانونية",
                lines=15,
                rtl=True,
                interactive=False,
            )

        with gr.TabItem("📄 المستندات المسترجعة"):
            docs_output = gr.Textbox(
                label="المستندات القانونية الأكثر صلة",
                lines=15,
                rtl=True,
                interactive=False,
            )

    # Actions
    submit_btn.click(
        fn=rag_interface,
        inputs=[question_input, top_k_slider],
        outputs=[answer_output, docs_output, status_output],
    )

    clear_btn.click(
        fn=lambda: ("", "", "", ""),
        outputs=[question_input, answer_output, docs_output, status_output],
    )

    question_input.submit(
        fn=rag_interface,
        inputs=[question_input, top_k_slider],
        outputs=[answer_output, docs_output, status_output],
    )


# Lancement de l'interface
print("🚀 Lancement de l'interface Gradio...")
demo.launch(share=True, debug=False)

🚀 Lancement de l'interface Gradio...
* Running on local URL:  http://127.0.0.1:7863


OSError: [WinError 225] Impossible de terminer l’opération, car le fichier contient un virus ou un logiciel potentiellement indésirable: 'C:\\Users\\hp\\.cache\\huggingface\\gradio\\frpc\\frpc_windows_amd64_v0.3'

---
## 9. 📝 Rapport Technique

### 9.1 Architecture du Système RAG

Notre système RAG est composé de cinq modules principaux :

**1. Module de Préparation des Données**  
Le CSV contient 529 articles du Code de la Route marocain en arabe. Le prétraitement normalise les caractères arabes (Alif, Ya, Waw), supprime les diacritiques, et découpe les articles longs en chunks de ≤500 caractères.

**2. Module Retriever Hybride**  
- **TF-IDF** (n-grammes de caractères 2-4) : Capture les correspondances lexicales exactes
- **SentenceTransformers** (paraphrase-multilingual-MiniLM-L12-v2) : Comprend la sémantique
- **FAISS** (IndexFlatIP) : Recherche des plus proches voisins efficace
- **RRF** : Fusion des deux listes par Reciprocal Rank Fusion

**3. Module Prompt Engineering**  
Le prompt est structuré avec rôle système (assistant juridique), contexte documentaire formaté, question, et instruction de citation obligatoire des articles.

**4. Module Générateur (LLM Simulé)**  
Sans API externe, le générateur extrait et reformule les informations clés des documents récupérés, cite les articles, et structure la réponse avec sanctions/amendes.

**5. Module Détection Hors Domaine**  
Combinaison de règles par mots-clés (domaine/hors-domaine) et d'un seuil de similarité sémantique (0.15) pour rejeter les questions non pertinentes.

### 9.2 Choix des Outils

| Outil | Choix | Raison |
|-------|-------|--------|
| SentenceTransformers | paraphrase-multilingual-MiniLM-L12-v2 | Multilingue, léger, supporte l'arabe |
| Vectorstore | FAISS | Rapide, open-source, pas de serveur requis |
| TF-IDF | scikit-learn char_wb 2-4 | Meilleur pour l'arabe (niveau caractère) |
| Fusion | RRF (k=60) | Standard industriel pour la recherche hybride |
| Interface | Gradio | Rapide à déployer, supporte RTL |

### 9.3 Analyse des Résultats

Le retriever hybride surpasse systématiquement TF-IDF seul et la recherche sémantique seule. La détection hors domaine atteint une précision élevée grâce à la combinaison de règles et de seuil sémantique. Le LLM simulé, bien que limité par rapport à un vrai modèle génératif, fournit des réponses structurées et traçables.

### 9.4 Limites et Perspectives

**Limites actuelles :**
- Le LLM simulé ne génère pas de langage naturel fluide
- Les scores ROUGE sont faibles (pas de vraie génération de texte)
- La détection hors domaine peut avoir des faux positifs

**Perspectives :**
- Intégration de Qwen2.5 local pour une vraie génération (recommandé)
- Fine-tuning sur le corpus juridique marocain
- Ajout de graphe de connaissances juridiques
- Déploiement sur Kaggle avec GPU

In [ ]:
# ─────────────────────────────────────────────
# Résumé final du système
# ─────────────────────────────────────────────
print("=" * 70)
print("📊 RÉSUMÉ FINAL DU SYSTÈME RAG")
print("=" * 70)
print(f"\n📁 Corpus : {len(df_clean)} articles juridiques du Code de la Route")
print(f"🧩 Chunks : {len(chunks_df)} unités indexées")
print(f"🔤 Vocabulaire TF-IDF : {tfidf_matrix.shape[1]} features")
print(f"🧠 Dimension embeddings : {embeddings.shape[1]}")
print(f"⚡ Vecteurs FAISS indexés : {faiss_index.ntotal}")
print(f"\n🔀 Stratégie de fusion : Reciprocal Rank Fusion (k=60)")
print(f"🚫 Détection hors domaine : Règles + Seuil sémantique (≥{SIMILARITY_THRESHOLD})")
print(f"\n✅ Système prêt à répondre aux questions juridiques sur le Code de la Route !")
print("=" * 70)